# Data Science & Database Assignment: Session 5
## Topic: WHERE Clause + LIKE Operator

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `restaurants.db`  
**Core Concepts:** Data Filtering (`WHERE`), Compound Logical Operators (`AND`, `OR`), Pattern Matching (`LIKE`, Wildcard `%`), Range Filtering (`BETWEEN ... AND`), Membership Testing (`IN`)  
**Status:** Completed & Pre-Executed  

---

### Filtering Paradigms in SQL

The `WHERE` clause forms the backbone of Data Query Language (DQL). It evaluates predicates row-by-row before any aggregation or projection occurs:

```text
                  PREDICATE FILTERING OPERATORS
                                │
       ┌────────────────────────┼────────────────────────┐
       ▼                        ▼                        ▼
Comparison & Logic       Pattern Matching         Set & Range
(>, <, =, AND, OR)      (LIKE, %, _)            (IN, BETWEEN)
```


## Task 1: Create `Restaurants` Table & Insert Zomato Sample Data
> **Problem Statement:** Create a table called `Restaurants` with columns: `id`, `name`, `cuisine`, `rating`, and `city`. Insert at least 5 sample records representing real or fictional restaurants you might find on Zomato.

### Schema Definition
```sql
CREATE TABLE Restaurants (
    id INT PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    cuisine VARCHAR(100) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL,
    city VARCHAR(100) NOT NULL
);
```


In [1]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("restaurants.db")
cursor = conn.cursor()

# Drop existing table for clean rerun
cursor.execute("DROP TABLE IF EXISTS Restaurants;")

# Create Restaurants table
cursor.execute("""
CREATE TABLE Restaurants (
    id INTEGER PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    cuisine VARCHAR(100) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL,
    city VARCHAR(100) NOT NULL
);
""")

# Insert 6 sample Zomato restaurants across Gujarat cities
sample_data = [
    (1, 'Swagat Restaurant', 'South Indian', 4.3, 'Ahmedabad'),
    (2, 'Swadisht Dining', 'North Indian', 4.1, 'Surat'),
    (3, 'Little Italy', 'Italian', 4.6, 'Ahmedabad'),
    (4, 'Mainland China', 'Chinese', 3.8, 'Vadodara'),
    (5, 'Sankalp', 'South Indian', 4.2, 'Surat'),
    (6, 'Cafe De Bistro', 'Continental', 3.2, 'Ahmedabad')
]

cursor.executemany("INSERT INTO Restaurants VALUES (?, ?, ?, ?, ?);", sample_data)
conn.commit()

df_all = pd.read_sql_query("SELECT * FROM Restaurants;", conn)
print("Task 1 Output: Populated 'Restaurants' Table")
display(df_all)


Task 1 Output: Populated 'Restaurants' Table


,id,name,cuisine,rating,city
0,1,Swagat Restaurant,South Indian,4.3,Ahmedabad
1,2,Swadisht Dining,North Indian,4.1,Surat
2,3,Little Italy,Italian,4.6,Ahmedabad
3,4,Mainland China,Chinese,3.8,Vadodara
4,5,Sankalp,South Indian,4.2,Surat
5,6,Cafe De Bistro,Continental,3.2,Ahmedabad


## Task 2: Multi-Condition Filtering (`rating > 4.0` in Ahmedabad or Surat)
> **Problem Statement:** Write a SQL query to find all restaurants in the `Restaurants` table that have a rating greater than 4.0 and are located in either `'Ahmedabad'` or `'Surat'`.

### SQL Query & Operator Precedence
```sql
SELECT * 
FROM Restaurants
WHERE rating > 4.0 
  AND (city = 'Ahmedabad' OR city = 'Surat');
```
* **Parentheses Precedence:** `AND` has higher precedence than `OR` in SQL. Wrapping `(city = 'Ahmedabad' OR city = 'Surat')` ensures that the city condition evaluates as a single logical block before being joined with `rating > 4.0`.
* **Equivalent `IN` Syntax:** `WHERE rating > 4.0 AND city IN ('Ahmedabad', 'Surat');`


In [2]:
query_t2 = """
SELECT * 
FROM Restaurants
WHERE rating > 4.0 
  AND (city = 'Ahmedabad' OR city = 'Surat');
"""

df_t2 = pd.read_sql_query(query_t2, conn)
print("Task 2 Output: Highly-Rated Restaurants (Rating > 4.0) in Ahmedabad or Surat")
display(df_t2)


Task 2 Output: Highly-Rated Restaurants (Rating > 4.0) in Ahmedabad or Surat


,id,name,cuisine,rating,city
0,1,Swagat Restaurant,South Indian,4.3,Ahmedabad
1,2,Swadisht Dining,North Indian,4.1,Surat
2,3,Little Italy,Italian,4.6,Ahmedabad
3,5,Sankalp,South Indian,4.2,Surat


## Task 3: Pattern Matching with `LIKE 'Swa%'`
> **Problem Statement:** Using the `LIKE` operator, write a query to select all restaurants whose names start with `'Swa'` (for example, `'Swagat'`, `'Swadisht'`) from the `Restaurants` table.  
> *Hint: Use `LIKE 'Swa%'`.*

### SQL Query & Wildcard Mechanics
```sql
SELECT * 
FROM Restaurants
WHERE name LIKE 'Swa%';
```
* **`%` Wildcard:** Represents zero, one, or multiple arbitrary characters.
* `'Swa%'` matches any string that begins with `'Swa'` followed by anything (e.g. `'Swagat Restaurant'`, `'Swadisht Dining'`), while rejecting `'Sankalp'` or `'Little Italy'`.


In [3]:
query_t3 = """
SELECT * 
FROM Restaurants
WHERE name LIKE 'Swa%';
"""

df_t3 = pd.read_sql_query(query_t3, conn)
print("Task 3 Output: Restaurants with Names Starting with 'Swa'")
display(df_t3)


Task 3 Output: Restaurants with Names Starting with 'Swa'


,id,name,cuisine,rating,city
0,1,Swagat Restaurant,South Indian,4.3,Ahmedabad
1,2,Swadisht Dining,North Indian,4.1,Surat


## Task 4: Range Filtering with `BETWEEN ... AND`
> **Problem Statement:** Write a SQL query using the `BETWEEN` keyword to find all restaurants in the `Restaurants` table with a rating between 3.5 and 4.5 (inclusive).

### SQL Query & Inclusivity
```sql
SELECT * 
FROM Restaurants
WHERE rating BETWEEN 3.5 AND 4.5;
```
* **Inclusive Boundary:** `BETWEEN val1 AND val2` is equivalent to `rating >= 3.5 AND rating <= 4.5`.
* Note that `'Little Italy'` (4.6) is excluded because it exceeds 4.5, and `'Cafe De Bistro'` (3.2) is excluded because it falls below 3.5.


In [4]:
query_t4 = """
SELECT * 
FROM Restaurants
WHERE rating BETWEEN 3.5 AND 4.5;
"""

df_t4 = pd.read_sql_query(query_t4, conn)
print("Task 4 Output: Restaurants with Rating in Range [3.5, 4.5]")
display(df_t4)


Task 4 Output: Restaurants with Rating in Range [3.5, 4.5]


,id,name,cuisine,rating,city
0,1,Swagat Restaurant,South Indian,4.3,Ahmedabad
1,2,Swadisht Dining,North Indian,4.1,Surat
2,4,Mainland China,Chinese,3.8,Vadodara
3,5,Sankalp,South Indian,4.2,Surat


## Task 5: Set Membership Filtering with `IN` Operator
> **Problem Statement:** Write a query to find all restaurants whose cuisine is either `'Chinese'`, `'Italian'`, or `'South Indian'` using the `IN` operator.

### SQL Query:
```sql
SELECT * 
FROM Restaurants
WHERE cuisine IN ('Chinese', 'Italian', 'South Indian');
```
* **The `IN` Operator:** Serves as a concise shorthand for multiple chained `OR` conditions:  
  `WHERE cuisine = 'Chinese' OR cuisine = 'Italian' OR cuisine = 'South Indian'`.
* Enhances query readability and allows query planners to use index hash lookups efficiently.


In [5]:
query_t5 = """
SELECT * 
FROM Restaurants
WHERE cuisine IN ('Chinese', 'Italian', 'South Indian');
"""

df_t5 = pd.read_sql_query(query_t5, conn)
print("Task 5 Output: Restaurants Matching Selected Cuisines (Chinese, Italian, South Indian)")
display(df_t5)

# Clean up connection
conn.close()


Task 5 Output: Restaurants Matching Selected Cuisines (Chinese, Italian, South Indian)


,id,name,cuisine,rating,city
0,1,Swagat Restaurant,South Indian,4.3,Ahmedabad
1,3,Little Italy,Italian,4.6,Ahmedabad
2,4,Mainland China,Chinese,3.8,Vadodara
3,5,Sankalp,South Indian,4.2,Surat


---

## 📌 Summary of Filtering Operators Mastered

| Operator | Function | Example Syntax |
| :--- | :--- | :--- |
| **`AND` / `OR`** | Combines multiple condition predicates | `WHERE rating > 4.0 AND (city = 'Ahmedabad' OR city = 'Surat')` |
| **`LIKE 'Swa%'`** | Prefix pattern matching | `WHERE name LIKE 'Swa%'` |
| **`LIKE '%x%'`** | Substring / containment matching | `WHERE name LIKE '%Cafe%'` |
| **`BETWEEN a AND b`** | Continuous range evaluation (inclusive) | `WHERE rating BETWEEN 3.5 AND 4.5` |
| **`IN (a, b, c)`** | Discrete set membership check | `WHERE cuisine IN ('Chinese', 'Italian', 'South Indian')` |
